# Inverse Heat Transfer with Automatic Differentiation

A steel plate is heating up. You have thermocouple readings at a handful of
locations, but you don't know the exact material properties — or even what the
temperature distribution looked like before heating started. Can you figure it out?

This notebook solves two inverse problems of increasing ambition using a Fortran
finite-difference solver whose exact derivatives are generated automatically by
[Enzyme](https://enzyme.mit.edu/) at the LLVM IR level — no manual adjoint code,
no source modifications. Then it shows how `jax.grad` can differentiate through
the Fortran solver end-to-end via [`tesseract-jax`](https://github.com/pasteurlabs/tesseract-jax).

1. **Part 1: Scalar calibration** — recover 2 material parameters (k₀, k₁) from 9 sensors
2. **Part 2: Thermal forensics** — recover the full 900-element initial temperature
   field from 100 sensors (finite differences would need 901 forward solves per
   iteration; one VJP gives all 900 gradients at once)
3. **Part 3: JAX integration** — `jax.grad` through Python → JAX → HTTP → Enzyme → Fortran
   in a single call

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from tesseract_core import Tesseract

# Grid and simulation parameters (fixed throughout)
nx, ny = 30, 30
n = nx * ny
n_steps = 500
dt = 0.05  # 500 steps x 0.05s = 25s (CFL-safe up to k0 ~ 80)
Lx, Ly = 0.1, 0.05

# Fixed physical parameters
rho = 7850.0  # density [kg/m^3] (mild steel)
cp = 460.0  # specific heat [J/(kg*K)]
h_conv = 25.0  # convection coefficient [W/(m^2*K)]
T_inf = 293.15  # ambient temperature [K]
T_hot = 373.15  # hot wall temperature [K]

# Initial condition: uniform ambient temperature
T_init = np.full(n, T_inf)
Q = np.zeros(n)  # no internal heating

print(f"Simulation: {n_steps} steps x {dt}s = {n_steps * dt:.0f}s total")

## Part 1: Scalar calibration (2 parameters)

### Generate synthetic data

We run the solver with the **true** material properties to produce a ground-truth
temperature field, then sample it at sparse sensor locations with added noise.
In practice, these would be thermocouple readings from a real component.

In [ ]:
# True material properties (what we want to recover)
k0_true = 45.0  # base conductivity [W/(m*K)]
k1_true = -0.02  # temperature coefficient [W/(m*K^2)]


def make_inputs(k0, k1):
    """Build a full input dict for the Tesseract."""
    return {
        "T_init": T_init,
        "Q": Q,
        "nx": nx,
        "ny": ny,
        "n_steps": n_steps,
        "k0": float(k0),
        "k1": float(k1),
        "rho": rho,
        "cp": cp,
        "h_conv": h_conv,
        "T_inf": T_inf,
        "T_hot": T_hot,
        "Lx": Lx,
        "Ly": Ly,
        "dt": dt,
    }


# Run the ground-truth simulation
with Tesseract.from_image("enzyme-thermal-2d:latest") as t:
    result_true = t.apply(inputs=make_inputs(k0_true, k1_true))
    T_true = np.array(result_true["T_final"]).reshape(ny, nx)

print(f"Ground truth: k0={k0_true}, k1={k1_true}")
print(f"Temperature range: {T_true.min():.2f} K to {T_true.max():.2f} K")

In [ ]:
# Place sensors on a regular grid in the interior (away from BCs)
# This mimics a realistic thermocouple layout
sensor_ix = [7, 15, 22]  # x positions
sensor_jy = [7, 15, 22]  # y positions
sensor_indices = []
sensor_coords = []

for jy in sensor_jy:
    for ix in sensor_ix:
        sensor_indices.append(jy * nx + ix)
        sensor_coords.append((ix, jy))

sensor_indices = np.array(sensor_indices)
n_sensors = len(sensor_indices)

# Extract true temperatures at sensor locations and add noise
rng = np.random.default_rng(42)
noise_std = 0.5  # K — realistic thermocouple noise
T_true_flat = T_true.flatten()
T_obs = T_true_flat[sensor_indices] + rng.normal(0, noise_std, n_sensors)

print(f"Number of sensors: {n_sensors}")
print(f"Noise std: {noise_std} K")
print(f"Observed temperatures: {T_obs.round(2)}")

In [ ]:
# Visualize the ground truth and sensor locations
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
im = ax.imshow(
    T_true, origin="lower", cmap="hot", extent=[0, Lx * 1e3, 0, Ly * 1e3], aspect="auto"
)
plt.colorbar(im, ax=ax, label="Temperature [K]")

# Plot sensor locations
for ix, jy in sensor_coords:
    x_mm = ix / (nx - 1) * Lx * 1e3
    y_mm = jy / (ny - 1) * Ly * 1e3
    ax.plot(x_mm, y_mm, "ws", markersize=8, markeredgecolor="blue", markeredgewidth=1.5)

ax.set_xlabel("x [mm]")
ax.set_ylabel("y [mm]")
ax.set_title(
    f"Ground truth temperature field (k₀={k0_true}, k₁={k1_true})\n□ = sensor locations"
)
plt.tight_layout()
plt.show()

## Step 2: Define the inverse problem

Minimize the misfit between predicted and observed sensor temperatures:

$$
J(k_0, k_1) = \frac{1}{2} \sum_{i=1}^{N_\text{sensors}} \left( T_\text{pred}(\mathbf{x}_i; k_0, k_1) - T_\text{obs}(\mathbf{x}_i) \right)^2
$$

The gradient $\nabla_{k_0, k_1} J$ comes from a single VJP (reverse-mode AD) call.
The cotangent vector is the residual at sensor locations, zero elsewhere:

$$
\bar{T}_j = \begin{cases}
T_\text{pred}(\mathbf{x}_j) - T_\text{obs}(\mathbf{x}_j) & \text{if } j \text{ is a sensor location} \\
0 & \text{otherwise}
\end{cases}
$$

One VJP call gives $\partial J / \partial k_0$ and $\partial J / \partial k_1$
simultaneously. Finite differences would need a separate forward solve per parameter.

In [ ]:
def objective_and_gradient(params, tesseract):
    """Compute loss and gradient for the inverse problem.

    params: [k0, k1]
    Returns: (loss, [dk0, dk1])
    """
    k0, k1 = params
    inputs = make_inputs(k0, k1)

    # Forward solve
    result = tesseract.apply(inputs=inputs)
    T_pred = np.array(result["T_final"])

    # Loss: sum of squared residuals at sensor locations
    residuals = T_pred[sensor_indices] - T_obs
    loss = 0.5 * np.sum(residuals**2)

    # Cotangent vector: residuals at sensor locations, zero elsewhere
    cotangent = np.zeros(n, dtype=np.float64)
    cotangent[sensor_indices] = residuals

    # One VJP call gives gradients w.r.t. both k0 and k1
    vjp = tesseract.vector_jacobian_product(
        inputs=inputs,
        vjp_inputs=["k0", "k1"],
        vjp_outputs=["T_final"],
        cotangent_vector={"T_final": cotangent},
    )

    grad = np.array([vjp["k0"], vjp["k1"]])
    return loss, grad

## Step 3: Run the optimization

We start from a deliberately wrong initial guess and use `scipy.optimize.minimize`
with L-BFGS-B (a quasi-Newton method that uses gradient information). The bounds
ensure physical plausibility (positive conductivity, reasonable temperature dependence).

In [ ]:
from scipy.optimize import minimize

# Initial guess: 30% off on k0, wrong sign on k1
k0_init = 60.0
k1_init = 0.01

# Track optimization history
history = {"k0": [k0_init], "k1": [k1_init], "loss": []}

with Tesseract.from_image("enzyme-thermal-2d:latest") as t:
    # Evaluate initial loss
    loss0, _ = objective_and_gradient([k0_init, k1_init], t)
    history["loss"].append(loss0)
    print(f"Initial guess: k0={k0_init:.2f}, k1={k1_init:.4f}, loss={loss0:.4f}")

    def callback(params):
        k0, k1 = params
        loss, _ = objective_and_gradient(params, t)
        history["k0"].append(k0)
        history["k1"].append(k1)
        history["loss"].append(loss)
        print(f"  k0={k0:.4f}, k1={k1:.6f}, loss={loss:.6f}")

    result = minimize(
        fun=lambda p: objective_and_gradient(p, t),
        x0=[k0_init, k1_init],
        method="L-BFGS-B",
        jac=True,  # objective_and_gradient returns (loss, grad)
        bounds=[(5.0, 80.0), (-0.08, 0.08)],  # stay CFL-safe
        callback=callback,
        options={"maxiter": 50, "ftol": 1e-12, "gtol": 1e-8},
    )

    # Final forward solve with recovered parameters
    k0_opt, k1_opt = result.x
    result_opt = t.apply(inputs=make_inputs(k0_opt, k1_opt))
    T_opt = np.array(result_opt["T_final"]).reshape(ny, nx)

print(f"\nOptimization finished in {result.nit} iterations")
print(f"  True:      k0={k0_true:.4f}, k1={k1_true:.6f}")
print(f"  Recovered: k0={k0_opt:.4f}, k1={k1_opt:.6f}")
print(
    f"  Error:     k0: {abs(k0_opt - k0_true) / k0_true * 100:.2f}%, "
    f"k1: {abs(k1_opt - k1_true) / abs(k1_true) * 100:.2f}%"
)

## Step 4: Visualize results

### Convergence

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Loss convergence
axes[0].semilogy(history["loss"], "k.-", linewidth=1.5)
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Loss (sum of squared residuals)")
axes[0].set_title("Convergence")
axes[0].grid(True, alpha=0.3)

# k0 convergence
axes[1].plot(history["k0"], "b.-", linewidth=1.5, label="k₀ estimate")
axes[1].axhline(
    k0_true, color="b", linestyle="--", alpha=0.5, label=f"k₀ true = {k0_true}"
)
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("k₀ [W/(m·K)]")
axes[1].set_title("Base conductivity")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# k1 convergence
axes[2].plot(history["k1"], "r.-", linewidth=1.5, label="k₁ estimate")
axes[2].axhline(
    k1_true, color="r", linestyle="--", alpha=0.5, label=f"k₁ true = {k1_true}"
)
axes[2].set_xlabel("Iteration")
axes[2].set_ylabel("k₁ [W/(m·K²)]")
axes[2].set_title("Temperature coefficient")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Temperature fields: initial guess vs. optimized vs. ground truth

In [ ]:
# Also compute the initial-guess temperature field for comparison
with Tesseract.from_image("enzyme-thermal-2d:latest") as t:
    result_init = t.apply(inputs=make_inputs(k0_init, k1_init))
    T_init_field = np.array(result_init["T_final"]).reshape(ny, nx)

# Common color range
vmin = min(T_true.min(), T_opt.min(), T_init_field.min())
vmax = max(T_true.max(), T_opt.max(), T_init_field.max())

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
extent = [0, Lx * 1e3, 0, Ly * 1e3]

im0 = axes[0].imshow(
    T_init_field,
    origin="lower",
    cmap="hot",
    extent=extent,
    aspect="auto",
    vmin=vmin,
    vmax=vmax,
)
axes[0].set_title(f"Initial guess\nk₀={k0_init}, k₁={k1_init}")

im1 = axes[1].imshow(
    T_opt,
    origin="lower",
    cmap="hot",
    extent=extent,
    aspect="auto",
    vmin=vmin,
    vmax=vmax,
)
axes[1].set_title(f"Recovered\nk₀={k0_opt:.2f}, k₁={k1_opt:.4f}")

im2 = axes[2].imshow(
    T_true,
    origin="lower",
    cmap="hot",
    extent=extent,
    aspect="auto",
    vmin=vmin,
    vmax=vmax,
)
axes[2].set_title(f"Ground truth\nk₀={k0_true}, k₁={k1_true}")

# Error field
error = np.abs(T_opt - T_true)
im3 = axes[3].imshow(error, origin="lower", cmap="Blues", extent=extent, aspect="auto")
axes[3].set_title(f"|Recovered - Truth|\nmax error: {error.max():.3f} K")

for ax in axes:
    ax.set_xlabel("x [mm]")
    ax.set_ylabel("y [mm]")
    # Mark sensor locations
    for ix, jy in sensor_coords:
        x_mm = ix / (nx - 1) * Lx * 1e3
        y_mm = jy / (ny - 1) * Ly * 1e3
        ax.plot(
            x_mm, y_mm, "ws", markersize=5, markeredgecolor="blue", markeredgewidth=1
        )

plt.colorbar(im2, ax=axes[:3].tolist(), label="Temperature [K]", shrink=0.9)
plt.colorbar(im3, ax=axes[3], label="Error [K]", shrink=0.9)
plt.tight_layout()
plt.show()

## Part 2: Thermal forensics — recovering a hidden heat signature (900 parameters)

A steel plate was subjected to an unmonitored heating event — say, a laser pulse
or a localized defect generating heat. Five seconds later, you measure temperatures
at 100 sensor locations. Can you reconstruct what the initial temperature
distribution looked like?

This is an ill-posed inverse problem: 900 unknowns (temperature at every grid cell)
from 100 noisy observations, through a nonlinear PDE. But with exact gradients from
Enzyme's VJP, L-BFGS-B handles it comfortably.

The cost advantage of reverse-mode AD is now dramatic:

| Method | Forward solves per iteration | Time per iteration |
|--------|----------------------------:|-------------------:|
| Finite differences | 901 (N+1) | ~10 s |
| VJP (reverse-mode AD) | 2 (fwd + rev) | ~0.8 s |
| **Speedup** | | **~450×** |

In [ ]:
# --- Part 2 setup ---
# Shorter simulation: 5 seconds (we want residual structure in the initial field)
n_steps_p2 = 100
dt_p2 = 0.05

# Fixed material properties (known — we're recovering T_init, not k)
k0_p2 = 45.0
k1_p2 = -0.01

# Build coordinate arrays for the 30x30 grid
x = np.linspace(0, Lx, nx)
y = np.linspace(0, Ly, ny)
X, Y = np.meshgrid(x, y)
X_flat, Y_flat = X.flatten(), Y.flatten()

# True initial temperature: two Gaussian hot spots on a warm background
T_init_true = (
    T_inf
    + 40.0 * np.exp(-((X_flat - 0.04) ** 2 + (Y_flat - 0.025) ** 2) / 0.015**2)
    + 25.0 * np.exp(-((X_flat - 0.08) ** 2 + (Y_flat - 0.035) ** 2) / 0.01**2)
)


# Run forward simulation with true initial field
def make_inputs_p2(T_init_field):
    return {
        "T_init": T_init_field.astype(np.float64),
        "Q": Q,
        "nx": nx,
        "ny": ny,
        "n_steps": n_steps_p2,
        "k0": k0_p2,
        "k1": k1_p2,
        "rho": rho,
        "cp": cp,
        "h_conv": h_conv,
        "T_inf": T_inf,
        "T_hot": T_hot,
        "Lx": Lx,
        "Ly": Ly,
        "dt": dt_p2,
    }


with Tesseract.from_image("enzyme-thermal-2d:latest") as t:
    result_true_p2 = t.apply(inputs=make_inputs_p2(T_init_true))
    T_final_true_p2 = np.array(result_true_p2["T_final"])

# 10x10 sensor grid (100 sensors in the interior)
sensor_ix_p2 = np.linspace(3, nx - 4, 10, dtype=int)
sensor_jy_p2 = np.linspace(3, ny - 4, 10, dtype=int)
sensor_grid = np.array([(jy * nx + ix) for jy in sensor_jy_p2 for ix in sensor_ix_p2])
n_sensors_p2 = len(sensor_grid)

# Observed data: true final temperatures at sensors + noise
noise_std_p2 = 0.3  # K
T_obs_p2 = T_final_true_p2[sensor_grid] + rng.normal(0, noise_std_p2, n_sensors_p2)

print(f"Grid: {nx}x{ny} = {n} unknowns")
print(f"Sensors: {n_sensors_p2}")
print(f"Simulation: {n_steps_p2} steps x {dt_p2}s = {n_steps_p2 * dt_p2:.0f}s")
print(f"True T_init range: {T_init_true.min():.1f} — {T_init_true.max():.1f} K")

In [ ]:
import time

from scipy.optimize import minimize as sp_minimize


def objective_and_gradient_p2(T_init_vec, tesseract):
    """Loss and gradient for the T_init recovery problem.

    900 unknowns, 100 observations, 1 VJP call for all 900 gradients.
    """
    inputs = make_inputs_p2(T_init_vec)

    # Forward solve
    result = tesseract.apply(inputs=inputs)
    T_pred = np.array(result["T_final"])

    # Sensor residuals
    residuals = T_pred[sensor_grid] - T_obs_p2
    loss = 0.5 * np.sum(residuals**2)

    # Cotangent: residuals at sensor locations
    cotangent = np.zeros(n, dtype=np.float64)
    cotangent[sensor_grid] = residuals

    # Single VJP gives gradient w.r.t. all 900 T_init components
    vjp = tesseract.vector_jacobian_product(
        inputs=inputs,
        vjp_inputs=["T_init"],
        vjp_outputs=["T_final"],
        cotangent_vector={"T_final": cotangent},
    )
    grad = np.array(vjp["T_init"])

    return loss, grad


# Run optimization: start from uniform ambient (the wrong answer)
T_init_guess = np.full(n, T_inf)
loss_history_p2 = []

with Tesseract.from_image("enzyme-thermal-2d:latest") as t:
    iter_count = [0]
    t_start = time.time()

    def callback_p2(x):
        iter_count[0] += 1
        if iter_count[0] % 10 == 0:
            loss, _ = objective_and_gradient_p2(x, t)
            loss_history_p2.append(loss)
            elapsed = time.time() - t_start
            print(f"  iter {iter_count[0]:3d}: loss={loss:.4f}, elapsed={elapsed:.1f}s")

    # Initial loss
    loss0, _ = objective_and_gradient_p2(T_init_guess, t)
    loss_history_p2.insert(0, loss0)
    print(f"Initial loss: {loss0:.2f}")
    print(f"Running L-BFGS-B with {n} parameters...")

    result_p2 = sp_minimize(
        fun=lambda x: objective_and_gradient_p2(x, t),
        x0=T_init_guess,
        method="L-BFGS-B",
        jac=True,
        bounds=[(250.0, 450.0)] * n,
        callback=callback_p2,
        options={"maxiter": 100, "ftol": 1e-14, "gtol": 1e-8},
    )

    elapsed_total = time.time() - t_start

    # Final loss
    loss_final, _ = objective_and_gradient_p2(result_p2.x, t)
    loss_history_p2.append(loss_final)

T_init_recovered = result_p2.x

print(f"\nOptimization finished: {result_p2.nit} iterations, {elapsed_total:.1f}s")
print(f"Loss: {loss0:.2f} → {loss_final:.4f}")
print(f"T_init correlation: {np.corrcoef(T_init_true, T_init_recovered)[0, 1]:.4f}")
print("\nCost comparison per iteration:")
print(
    f"  Finite differences: {n + 1} forward solves = ~{(n + 1) * elapsed_total / result_p2.nfev:.1f}s"
)
print(
    f"  Reverse-mode AD:    2 solves (fwd+rev)   = ~{2 * elapsed_total / result_p2.nfev:.2f}s"
)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

extent = [0, Lx * 1e3, 0, Ly * 1e3]

# --- Top row: initial temperature fields ---
vmin_init = min(T_init_true.min(), T_init_recovered.min(), T_inf)
vmax_init = max(T_init_true.max(), T_init_recovered.max())

axes[0, 0].imshow(
    T_init_guess.reshape(ny, nx),
    origin="lower",
    cmap="hot",
    extent=extent,
    aspect="auto",
    vmin=vmin_init,
    vmax=vmax_init,
)
axes[0, 0].set_title("Starting guess\n(uniform ambient)")

axes[0, 1].imshow(
    T_init_recovered.reshape(ny, nx),
    origin="lower",
    cmap="hot",
    extent=extent,
    aspect="auto",
    vmin=vmin_init,
    vmax=vmax_init,
)
axes[0, 1].set_title(
    f"Recovered T₀\n(corr={np.corrcoef(T_init_true, T_init_recovered)[0, 1]:.3f})"
)

im_true = axes[0, 2].imshow(
    T_init_true.reshape(ny, nx),
    origin="lower",
    cmap="hot",
    extent=extent,
    aspect="auto",
    vmin=vmin_init,
    vmax=vmax_init,
)
axes[0, 2].set_title("True T₀\n(two Gaussian hot spots)")

plt.colorbar(im_true, ax=axes[0, :].tolist(), label="Temperature [K]", shrink=0.85)

# Mark sensor locations on all top-row plots
for ax in axes[0, :]:
    for jy_idx in sensor_jy_p2:
        for ix_idx in sensor_ix_p2:
            x_mm = ix_idx / (nx - 1) * Lx * 1e3
            y_mm = jy_idx / (ny - 1) * Ly * 1e3
            ax.plot(x_mm, y_mm, ".", color="cyan", markersize=2, alpha=0.5)
    ax.set_xlabel("x [mm]")
    ax.set_ylabel("y [mm]")

# --- Bottom row: diagnostics ---

# Recovery error
error_p2 = np.abs(T_init_recovered - T_init_true).reshape(ny, nx)
im_err = axes[1, 0].imshow(
    error_p2, origin="lower", cmap="Blues", extent=extent, aspect="auto"
)
axes[1, 0].set_title(
    f"|Recovered - True|\nmax={error_p2.max():.1f} K, mean={error_p2.mean():.1f} K"
)
axes[1, 0].set_xlabel("x [mm]")
axes[1, 0].set_ylabel("y [mm]")
plt.colorbar(im_err, ax=axes[1, 0], label="Error [K]", shrink=0.85)

# Scatter: true vs recovered
axes[1, 1].scatter(T_init_true, T_init_recovered, s=3, alpha=0.5, c="steelblue")
lims = [vmin_init - 5, vmax_init + 5]
axes[1, 1].plot(lims, lims, "k--", alpha=0.5, label="perfect recovery")
axes[1, 1].set_xlim(lims)
axes[1, 1].set_ylim(lims)
axes[1, 1].set_xlabel("True T₀ [K]")
axes[1, 1].set_ylabel("Recovered T₀ [K]")
axes[1, 1].set_title("True vs. recovered (per grid cell)")
axes[1, 1].legend()
axes[1, 1].set_aspect("equal")
axes[1, 1].grid(True, alpha=0.3)

# Convergence
axes[1, 2].semilogy(loss_history_p2, "k.-", linewidth=1.5)
axes[1, 2].set_xlabel("Checkpoint")
axes[1, 2].set_ylabel("Loss")
axes[1, 2].set_title(f"Convergence ({result_p2.nit} L-BFGS iterations)")
axes[1, 2].grid(True, alpha=0.3)

plt.suptitle(
    "Part 2: Recovering a 900-element initial temperature field from 100 sensors",
    fontsize=13,
    fontweight="bold",
    y=1.01,
)
plt.tight_layout()
plt.show()

With 2 parameters, finite differences are still practical. With 900, reverse-mode
AD is ~450× faster. The gap only widens on production meshes. But so far we've been
calling `apply` and `vector_jacobian_product` manually — can we do better?

## Part 3: End-to-end JAX autodiff through a Fortran solver

With [`tesseract-jax`](https://github.com/pasteurlabs/tesseract-jax),
the Tesseract becomes a JAX primitive. `jax.grad` just works — through six layers:

```
jax.grad (Python)
  └─ JAX reverse-mode AD
       └─ apply_tesseract (JAX primitive)
            └─ HTTP/JSON call to Tesseract container
                 └─ vector_jacobian_product endpoint
                      └─ Enzyme reverse-mode AD (LLVM IR)
                           └─ Fortran solver (thermal_2d_solve)
```

In [ ]:
import jax
import jax.numpy as jnp
from tesseract_jax import apply_tesseract

# Serve both Tesseracts (they stay alive for the rest of the notebook)
enzyme_tess = Tesseract.from_image("enzyme-thermal-2d:latest")
enzyme_tess.serve()

jax_tess = Tesseract.from_image("jax-thermal-2d:latest")
jax_tess.serve()

# Build inputs as JAX arrays (required by apply_tesseract)
jax_inputs = {
    "T_init": jnp.full(n, T_inf, dtype=jnp.float64),
    "Q": jnp.zeros(n, dtype=jnp.float64),
    "nx": nx,
    "ny": ny,
    "n_steps": n_steps,
    "k0": jnp.float64(45.0),
    "k1": jnp.float64(-0.02),
    "rho": jnp.float64(rho),
    "cp": jnp.float64(cp),
    "h_conv": jnp.float64(h_conv),
    "T_inf": jnp.float64(T_inf),
    "T_hot": jnp.float64(T_hot),
    "Lx": jnp.float64(Lx),
    "Ly": jnp.float64(Ly),
    "dt": jnp.float64(dt),
}

# The Fortran solver is now a JAX-differentiable function
T_final = apply_tesseract(enzyme_tess, jax_inputs)["T_final"]
print(
    f"Forward pass through Fortran solver: T range [{T_final.min():.2f}, {T_final.max():.2f}]"
)
print(f"Type: {type(T_final)} — it's a JAX array")

In [ ]:
# Define a loss function that mixes local JAX ops with a remote Fortran solver.
# jax.grad differentiates through all of it, including the HTTP call to Enzyme.


def loss_fn(k0, k1, tesseract):
    """Sensor misfit loss. Calls a Fortran solver via apply_tesseract."""
    inputs = {**jax_inputs, "k0": k0, "k1": k1}
    T_pred = apply_tesseract(tesseract, inputs)["T_final"]
    residuals = T_pred[sensor_indices] - jnp.array(T_obs)
    return 0.5 * jnp.sum(residuals**2)


k0_test = jnp.float64(60.0)
k1_test = jnp.float64(0.01)

# Enzyme backend: JAX AD → HTTP → Enzyme AD → Fortran
loss_enzyme, (dk0_enzyme, dk1_enzyme) = jax.value_and_grad(loss_fn, argnums=(0, 1))(
    k0_test, k1_test, enzyme_tess
)
print("Enzyme backend (Fortran):")
print(f"  loss = {loss_enzyme:.4f}")
print(f"  ∂loss/∂k0 = {dk0_enzyme:.6f}")
print(f"  ∂loss/∂k1 = {dk1_enzyme:.6f}")

# JAX backend: JAX AD → HTTP → jax.vjp → XLA
loss_jax, (dk0_jax, dk1_jax) = jax.value_and_grad(loss_fn, argnums=(0, 1))(
    k0_test, k1_test, jax_tess
)
print("\nJAX backend (Python):")
print(f"  loss = {loss_jax:.4f}")
print(f"  ∂loss/∂k0 = {dk0_jax:.6f}")
print(f"  ∂loss/∂k1 = {dk1_jax:.6f}")

print(
    f"\nGradient agreement: Δdk0 = {abs(dk0_enzyme - dk0_jax):.2e}, "
    f"Δdk1 = {abs(dk1_enzyme - dk1_jax):.2e}"
)

In [ ]:
# Full optimization loop using jax.value_and_grad.
# The optimizer just sees a JAX function.

k0_opt_jax = jnp.float64(60.0)
k1_opt_jax = jnp.float64(0.01)
lr = jnp.float64(0.5)

loss_history_p3 = []

grad_fn = jax.value_and_grad(loss_fn, argnums=(0, 1))

print("Gradient descent through Fortran solver via tesseract-jax")
print(f"{'iter':>4s}  {'loss':>10s}  {'k0':>8s}  {'k1':>10s}")
print("-" * 40)

for i in range(20):
    loss_val, (dk0, dk1) = grad_fn(k0_opt_jax, k1_opt_jax, enzyme_tess)
    loss_history_p3.append(float(loss_val))

    k0_opt_jax = jnp.clip(k0_opt_jax - lr * dk0, 5.0, 80.0)
    k1_opt_jax = jnp.clip(k1_opt_jax - lr * dk1, -0.08, 0.08)

    if i % 5 == 0 or i == 19:
        print(f"{i:4d}  {loss_val:10.4f}  {k0_opt_jax:8.4f}  {k1_opt_jax:10.6f}")

print(f"\nRecovered: k0={float(k0_opt_jax):.4f}, k1={float(k1_opt_jax):.6f}")
print(f"True:      k0={k0_true:.4f}, k1={k1_true:.6f}")

In [ ]:
# Clean up
enzyme_tess.teardown()
jax_tess.teardown()

## What just happened

A single `jax.value_and_grad` call triggered this chain:

| Layer | Technology | Role |
|-------|-----------|------|
| Optimizer | Python / JAX | Gradient descent loop |
| AD framework | JAX reverse-mode | Propagates cotangents |
| Tesseract bridge | `tesseract-jax` | Registers JAX primitive, dispatches HTTP calls |
| Transport | HTTP + JSON | Crosses process/container boundary |
| AD engine | Enzyme (LLVM pass) | Generates VJP from compiled Fortran IR |
| Solver | Fortran 90 | `thermal_2d_solve` |

Swapping `enzyme_tess` for `jax_tess` produces the same gradients — the
optimization code doesn't change.

The Fortran solver was never modified. Enzyme differentiated it from the compiled
LLVM IR. Tesseract made it callable — and differentiable — from JAX.